In [1]:
import os
import zipfile
import pickle
import pandas as pd
from tqdm import tqdm  # Import tqdm for the progress bar

def load_dataframe_from_zip(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        # Iterate through all files in the zip
        for file_name in zip_ref.namelist():
            if file_name.endswith('.pkl'):
                with zip_ref.open(file_name) as file:
                    # Load the data from the pickle file
                    data = pickle.load(file)
                    
                    # Check if the data is a list, and if so, convert it to a DataFrame
                    if isinstance(data, list):
                        df = pd.DataFrame(data, columns=['cik', 'date', 'note_text'])
                        df['item_7'] = df['note_text'].apply(extract_item_7)
                        df['risk_factors'] = df['note_text'].apply(extract_risk_factors)
                        return df[['cik', 'date','item_7','risk_factors']]
                    else:
                        print(f"Warning: The file {file_name} does not contain a list or DataFrame.")
    return None

def combine_pickles_in_folders(folders):
    combined_df = pd.DataFrame()
    
    # Loop through each folder
    for folder in folders:
        for root, dirs, files in os.walk(folder):
            # Add tqdm to show progress when iterating over the files
            for file in tqdm(files, desc=f"Processing files in {folder}", unit="file"):
                if file.endswith('.zip'):
                    zip_path = os.path.join(root, file)
                    # Load the dataframe from the zip file
                    df = load_dataframe_from_zip(zip_path)
                    if df is not None:
                        combined_df = pd.concat([combined_df, df], ignore_index=True)
    
    return combined_df

def save_as_multiple_parquet(df, output_folder, num_files=10):
    # Split the DataFrame into `num_files` chunks
    chunk_size = len(df) // num_files
    for i in range(num_files):
        # Determine start and end index for each chunk
        start_idx = i * chunk_size
        end_idx = (i + 1) * chunk_size if i < num_files - 1 else len(df)
        
        # Create a chunk DataFrame
        chunk_df = df.iloc[start_idx:end_idx]
        
        # Define the file name
        output_path = os.path.join(output_folder, f"combined_part_{i+1}.parquet")
        
        # Save the chunk as a Parquet file
        chunk_df.to_parquet(output_path, compression='gzip', index=False)
        print(f"Saved chunk {i+1} as {output_path}")

# Define the folders to search
folders = ['../processed_data/processed_data_new', '../processed_data/processed_data']

# Combine the dataframes
combined_dataframe = combine_pickles_in_folders(folders)

# Save the combined dataframe as multiple Parquet files
if not combined_dataframe.empty:
    output_folder = 'output'  # Replace with your desired output folder
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    save_as_multiple_parquet(combined_dataframe, output_folder, num_files=10)
else:
    print("No data found to combine.")


Processing files in ../processed_data/processed_data_new:   0%|                               | 0/41 [00:03<?, ?file/s]


NameError: name 'extract_item_7' is not defined